# Trabalho IAA011 — Visão Computacional

## Exercício 1: extração de características

Os bancos de imagens fornecidos são conjuntos de imagens de 250x250 pixels de imuno-histoquímica 
(biópsia) de câncer de mama. No total são 4 classes (0, 1+, 2+ e 3+) que estão divididas em diretórios. O 
objetivo é classificar as imagens nas categorias correspondentes. Uma base de imagens será utilizada para 
o treinamento e outra para o teste do treino.

### Passo 1 — Importações e configurações iniciais


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from skimage.feature import local_binary_pattern
from sklearn.model_selection import StratifiedGroupKFold
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.models import Model

Configuração de Pastas e Parâmetros

In [2]:
# Local das pastas de treino e teste, e da pasta de saída.
TRAIN_DIR = Path("Train_Warwick\Train_4cls_amostra")
TEST_DIR = Path("Test_Warwick\Test_4cl_amostra")
OUTPUT_DIR = Path("saidas_exercicio1") # Local onde serão salvos os CSVs e as saídas do exercício.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Extensões consideradas como imagens.
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

# Configuração LBP usada neste trabalho: 8 vizinhos e raio 1.
LBP_P = 8
LBP_R = 1
LBP_METHOD = "uniform"
N_LBP_BINS = LBP_P + 2  # para método uniform, são P + 2 possíveis códigos

print("Pasta treino existe:", TRAIN_DIR.exists(), "->", TRAIN_DIR)
print("Pasta teste existe:", TEST_DIR.exists(), "->", TEST_DIR)

Pasta treino existe: True -> Train_Warwick\Train_4cls_amostra
Pasta teste existe: True -> Test_Warwick\Test_4cl_amostra


<>:2: SyntaxWarning: invalid escape sequence '\T'
<>:3: SyntaxWarning: invalid escape sequence '\T'
<>:2: SyntaxWarning: invalid escape sequence '\T'
<>:3: SyntaxWarning: invalid escape sequence '\T'
C:\Users\gusta\AppData\Local\Temp\ipykernel_23072\466055809.py:2: SyntaxWarning: invalid escape sequence '\T'
  TRAIN_DIR = Path("Train_Warwick\Train_4cls_amostra")
C:\Users\gusta\AppData\Local\Temp\ipykernel_23072\466055809.py:3: SyntaxWarning: invalid escape sequence '\T'
  TEST_DIR = Path("Test_Warwick\Test_4cl_amostra")


### Passo 2 — Carregar as imagens e identificar os pacientes

A função percorre as subpastas das classes e cria uma tabela com o caminho, nome do arquivo, identificador do paciente e rótulo. O ID é o texto antes do primeiro `_`, por exemplo, `01` em `01_HER2_0001.png`. Isso permite separar os pacientes sem misturar suas imagens.

In [3]:
def obter_id_paciente(nome_arquivo):
    partes = Path(nome_arquivo).stem.split("_")
    if len(partes) < 3:
        raise ValueError(f"Nome de arquivo fora do padrão esperado: {nome_arquivo}")
    return partes[0]

def carregar_manifesto(pasta_base):
    if not pasta_base.is_dir():
        raise FileNotFoundError(f"Pasta não encontrada: {pasta_base.resolve()}")

    linhas = []
    for pasta_classe in sorted(p for p in pasta_base.iterdir() if p.is_dir()):
        for arquivo in sorted(pasta_classe.rglob("*")):
            if arquivo.is_file() and arquivo.suffix.lower() in IMAGE_EXTENSIONS:
                linhas.append({
                    "image_path": str(arquivo.resolve()),
                    "filename": arquivo.name,
                    "patient_id": obter_id_paciente(arquivo.name),
                    "label": pasta_classe.name,
                })

    if not linhas:
        raise ValueError(f"Nenhuma imagem encontrada em {pasta_base}")
    return pd.DataFrame(linhas)


df_treino_original = carregar_manifesto(TRAIN_DIR)
df_teste = carregar_manifesto(TEST_DIR)

print("Imagens na base de treino:", len(df_treino_original))
print("Pacientes na base de treino:", df_treino_original["patient_id"].nunique())
print("Imagens na base de teste:", len(df_teste))
print("Pacientes na base de teste:", df_teste["patient_id"].nunique())
print("Classes no treino:", sorted(df_treino_original["label"].unique()))
print("Classes no teste:", sorted(df_teste["label"].unique()))
display(df_treino_original.head())

Imagens na base de treino: 593
Pacientes na base de treino: 20
Imagens na base de teste: 371
Pacientes na base de teste: 13
Classes no treino: ['0', '1', '2', '3']
Classes no teste: ['0', '1', '2', '3']


,image_path,filename,patient_id,label
0,C:\workspace\ufpr\IAA-2026\TRABALHO_DE_IAA011\...,01_HER2_2775.png,01,0
1,C:\workspace\ufpr\IAA-2026\TRABALHO_DE_IAA011\...,01_HER2_3372.png,01,0
2,C:\workspace\ufpr\IAA-2026\TRABALHO_DE_IAA011\...,01_HER2_3570.png,01,0
3,C:\workspace\ufpr\IAA-2026\TRABALHO_DE_IAA011\...,01_HER2_3973.png,01,0
4,C:\workspace\ufpr\IAA-2026\TRABALHO_DE_IAA011\...,01_HER2_4153.png,01,0


### Passo 3 — Divide os dados de treinamento em treinamento 80% e validação 20%
Foi dividido a base em cinco partes mantendo junto imagens do mesmo paciente

In [4]:
# Cria 5 divisões, mantendo as imagens de cada paciente juntas
divisor = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

indices_treino, indices_validacao = next(
    divisor.split(
        df_treino_original,
        df_treino_original["label"],
        groups=df_treino_original["patient_id"]
    )
)

df_treino = df_treino_original.iloc[indices_treino].copy()
df_validacao = df_treino_original.iloc[indices_validacao].copy()

# Garante que nenhum paciente aparece nos dois conjuntos.
assert set(df_treino["patient_id"]).isdisjoint(set(df_validacao["patient_id"]))

print("Divisão concluída!")
print("Imagens de treino:", len(df_treino))
print("Imagens de validação:", len(df_validacao))

Divisão concluída!
Imagens de treino: 476
Imagens de validação: 117


### Passo 4 — Função de extração LBP e Salvar CSV

LBP compara cada pixel com seus vizinhos para descrever a textura. Para cada imagem, calculamos o LBP em escala de cinza e contamos a frequência das 10 categorias. O histograma é normalizado para que seus valores somem 1.

In [5]:
def extrair_lbp(caminho_imagem):
    with Image.open(caminho_imagem) as imagem:
        imagem_cinza = np.asarray(imagem.convert("L"), dtype=np.uint8)

    lbp = local_binary_pattern(
        imagem_cinza, P=LBP_P, R=LBP_R, method=LBP_METHOD
    )
    histograma = np.bincount(
        lbp.astype(np.int32).ravel(), minlength=N_LBP_BINS
    )[:N_LBP_BINS].astype(np.float32)

    return histograma / histograma.sum()

Cada linha do CSV representa uma imagem; as primeiras colunas guardam os metadados e as 10 colunas seguintes guardam as características LBP.

In [6]:
def gerar_csv_lbp(df, nome_particao):
    linhas = []
    for registro in df.itertuples(index=False):
        vetor = extrair_lbp(registro.image_path)
        linhas.append([
            registro.image_path, registro.filename,
            registro.patient_id, registro.label, *vetor
        ])

    colunas = ["image_path", "filename", "patient_id", "label"]
    colunas += [f"lbp_{i}" for i in range(N_LBP_BINS)]
    resultado = pd.DataFrame(linhas, columns=colunas)
    resultado.to_csv(OUTPUT_DIR / f"features_lbp_{nome_particao}.csv", index=False)
    print(f"LBP {nome_particao}: {resultado.shape}")
    return resultado


df_lbp_treino = gerar_csv_lbp(df_treino, "treino")
df_lbp_validacao = gerar_csv_lbp(df_validacao, "validacao")
df_lbp_teste = gerar_csv_lbp(df_teste, "teste")
display(df_lbp_treino.head())

LBP treino: (476, 14)
LBP validacao: (117, 14)
LBP teste: (371, 14)


,image_path,filename,patient_id,label,lbp_0,lbp_1,lbp_2,lbp_3,lbp_4,lbp_5,lbp_6,lbp_7,lbp_8,lbp_9
0,C:\workspace\ufpr\IAA-2026\TRABALHO_DE_IAA011\...,01_HER2_2775.png,01,0,0.011856,0.031728,0.040160,0.154640,0.370224,0.215056,0.069376,0.033872,0.030752,0.042336
1,C:\workspace\ufpr\IAA-2026\TRABALHO_DE_IAA011\...,01_HER2_3372.png,01,0,0.015280,0.035856,0.044240,0.152272,0.345648,0.205952,0.072048,0.035904,0.044496,0.048304
2,C:\workspace\ufpr\IAA-2026\TRABALHO_DE_IAA011\...,01_HER2_3570.png,01,0,0.017600,0.039088,0.049232,0.154240,0.314112,0.204544,0.079920,0.039856,0.046352,0.055056
3,C:\workspace\ufpr\IAA-2026\TRABALHO_DE_IAA011\...,01_HER2_3973.png,01,0,0.013312,0.034544,0.037296,0.140656,0.332304,0.215808,0.071776,0.038288,0.068656,0.047360
4,C:\workspace\ufpr\IAA-2026\TRABALHO_DE_IAA011\...,01_HER2_4153.png,01,0,0.015408,0.037168,0.044640,0.152416,0.340352,0.208320,0.075760,0.037888,0.037472,0.050576


### Passo 5 — Carregar a VGG16 pré-treinada

Agora usamos a VGG16 com pesos do ImageNet. `include_top=True` mantém as camadas finais originais e escolhemos `fc2`, a penúltima camada, como saída. Ela gera um vetor de 4096 características por imagem. Os pesos ficam congelados porque estamos apenas extraindo características.

In [7]:
vgg16_base = VGG16(weights="imagenet", include_top=True, input_shape=(224, 224, 3))
vgg16_extrator = Model(
    inputs=vgg16_base.input,
    outputs=vgg16_base.get_layer("fc2").output
)
vgg16_extrator.trainable = False

VGG_SIZE = (224, 224)
VGG_BATCH_SIZE = 16
VGG_N_FEATURES = 4096

print("VGG16 pronta para extrair", VGG_N_FEATURES, "características por imagem.")

553467096/553467096 ━━━━━━━━━━━━━━━━━━━━ 18s 0us/step
VGG16 pronta para extrair 4096 características por imagem.


### Passo 6 — Preparar imagens e extrair características VGG16

A VGG16 espera imagens RGB de 224 × 224 pixels. A função abaixo redimensiona e aplica o pré-processamento esperado pelos pesos do ImageNet.

In [8]:
def preparar_imagem_vgg16(caminho_imagem):
    with Image.open(caminho_imagem) as imagem:
        imagem = imagem.convert("RGB").resize(VGG_SIZE)
        imagem = np.asarray(imagem, dtype=np.float32)
    return preprocess_input(imagem)


def gerar_csv_vgg16(df, nome_particao):
    lotes = []
    for inicio in range(0, len(df), VGG_BATCH_SIZE):
        lote_df = df.iloc[inicio:inicio + VGG_BATCH_SIZE]
        imagens = np.stack([
            preparar_imagem_vgg16(caminho)
            for caminho in lote_df["image_path"]
        ])
        lotes.append(vgg16_extrator.predict(imagens, verbose=0))
        print(f"VGG16 {nome_particao}: {min(inicio + VGG_BATCH_SIZE, len(df))}/{len(df)}")

    caracteristicas = np.concatenate(lotes, axis=0)
    colunas = [f"vgg16_{i}" for i in range(VGG_N_FEATURES)]
    df_caracteristicas = pd.DataFrame(caracteristicas, columns=colunas)
    metadados = df[["image_path", "filename", "patient_id", "label"]].reset_index(drop=True)
    resultado = pd.concat([metadados, df_caracteristicas], axis=1)
    resultado.to_csv(
        OUTPUT_DIR / f"features_vgg16_{nome_particao}.csv",
        index=False, float_format="%.7g"
    )
    print(f"CSV VGG16 {nome_particao}: {resultado.shape}")
    return resultado

In [9]:
df_vgg16_treino = gerar_csv_vgg16(df_treino, "treino")
df_vgg16_validacao = gerar_csv_vgg16(df_validacao, "validacao")
df_vgg16_teste = gerar_csv_vgg16(df_teste, "teste")

print("\nArquivos gerados:")
for arquivo in sorted(OUTPUT_DIR.glob("*.csv")):
    print(arquivo.name)

VGG16 treino: 16/476
VGG16 treino: 32/476
VGG16 treino: 48/476
VGG16 treino: 64/476
VGG16 treino: 80/476
VGG16 treino: 96/476
VGG16 treino: 112/476
VGG16 treino: 128/476
VGG16 treino: 144/476
VGG16 treino: 160/476
VGG16 treino: 176/476
VGG16 treino: 192/476
VGG16 treino: 208/476
VGG16 treino: 224/476
VGG16 treino: 240/476
VGG16 treino: 256/476
VGG16 treino: 272/476
VGG16 treino: 288/476
VGG16 treino: 304/476
VGG16 treino: 320/476
VGG16 treino: 336/476
VGG16 treino: 352/476
VGG16 treino: 368/476
VGG16 treino: 384/476
VGG16 treino: 400/476
VGG16 treino: 416/476
VGG16 treino: 432/476
VGG16 treino: 448/476
VGG16 treino: 464/476
VGG16 treino: 476/476
CSV VGG16 treino: (476, 4100)
VGG16 validacao: 16/117
VGG16 validacao: 32/117
VGG16 validacao: 48/117
VGG16 validacao: 64/117
VGG16 validacao: 80/117
VGG16 validacao: 96/117
VGG16 validacao: 112/117
VGG16 validacao: 117/117
CSV VGG16 validacao: (117, 4100)
VGG16 teste: 16/371
VGG16 teste: 32/371
VGG16 teste: 48/371
VGG16 teste: 64/371
VGG16 tes